# Lab 7 · Notebook 2: Showing it

**Time:** about 40 minutes. **What you will do:** turn the SOX10 result from notebook 1 into figures, starting from an empty figure and ending with a box plot that shows every cell line as a dot.

Every **Write it** cell asks you to make a figure called `fig` with a panel called `ax`. The self-check underneath looks at `ax` to see what you drew, so keep those two names.

In [ ]:
# Run this first. It loads the DepMap table and makes skin and rest.
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

FILE = "depmap.csv"
URL = "https://raw.githubusercontent.com/HocheggerLab/y3-bio-python/main/lab07/data/" + FILE

if not Path(FILE).exists():
    urllib.request.urlretrieve(URL, FILE)

df = pd.read_csv(FILE)
skin = df[df["oncotree_lineage"] == "Skin"]
rest = df[df["oncotree_lineage"] != "Skin"]
print("all:", len(df), "  skin:", len(skin), "  rest:", len(rest))

## 1. An empty figure

Every figure in this course starts with the same line:

```python
fig, ax = plt.subplots()
```

`fig` is the sheet of paper. `ax` is one panel on it, with its two axes. You draw data on `ax`, and you label `ax`. `plt.show()` displays the result.

In [ ]:
# DEMO - a figure with labels but no data yet
fig, ax = plt.subplots()
ax.set_xlabel("x label goes here")
ax.set_ylabel("y label goes here")
ax.set_title("A title for the panel")
plt.show()

### Write it

**Your task:** make an empty figure whose x label is `BRAF gene effect`, whose y label is `SOX10 gene effect`, and whose title is `Two genes in 1,165 cell lines`.

In [ ]:
# TODO - an empty figure with three labels



# --- self-check (do not edit) ---
assert ax.get_xlabel() == "BRAF gene effect", f"The x label is {ax.get_xlabel()!r}. Use ax.set_xlabel(...)"
assert ax.get_ylabel() == "SOX10 gene effect", f"The y label is {ax.get_ylabel()!r}. Use ax.set_ylabel(...)"
assert ax.get_title() == "Two genes in 1,165 cell lines", f"The title is {ax.get_title()!r}. Use ax.set_title(...)"
print("Correct: three labels on an empty panel")

## 2. One column: a histogram

A histogram shows how the values in **one column** are spread out. `ax.hist` cuts the range into bins and draws one bar per bin; the height of a bar is the number of cell lines in it.

In [ ]:
# DEMO - SOX10 across all 1,165 cell lines
fig, ax = plt.subplots()
ax.hist(df["SOX10"], bins=40)
ax.set_xlabel("SOX10 gene effect")
ax.set_ylabel("Number of cell lines")
plt.show()

Most cell lines sit near 0: they do not need SOX10. A small group trails off to the left, down to about -3. Notebook 1 told you who they are.

### Write it

**Predict first:** BRAF is needed by melanoma, and by a few other cancers. Will its histogram look like SOX10's?

**Your task:** draw a histogram of `BRAF` across **all** cell lines with **30** bins. Label both axes.

In [ ]:
# TODO - a histogram of BRAF



# --- self-check (do not edit) ---
assert len(ax.patches) == 30, f"Your histogram has {len(ax.patches)} bars. Use bins=30"
total = sum(bar.get_height() for bar in ax.patches)
assert total == 1165, f"Your bars add up to {total:.0f} cell lines, not 1,165. Use the whole table, df"
assert abs(ax.patches[0].get_x() - df["BRAF"].min()) < 1e-9, "The bars do not start at the lowest BRAF value. Did you plot the BRAF column?"
assert "BRAF" in ax.get_xlabel(), "Give the x axis a label that says which gene it is"
assert ax.get_ylabel() != "", "Give the y axis a label"
print("Correct: 30 bins, all 1,165 cell lines")

### Write it

**Your task:** now draw a histogram of `SOX10` in the **skin** lines only, with 20 bins. Label both axes and give it a title.

Look at the shape. Is it one hump, or two? Think of the 18 lines from notebook 1.

In [ ]:
# TODO - a histogram of SOX10 in skin



# --- self-check (do not edit) ---
assert len(ax.patches) == 20, f"Your histogram has {len(ax.patches)} bars. Use bins=20"
total = sum(bar.get_height() for bar in ax.patches)
assert total == 73, f"Your bars add up to {total:.0f} cell lines. Plot skin['SOX10'], the 73 skin lines"
assert ax.get_xlabel() and ax.get_ylabel() and ax.get_title(), "Add an x label, a y label and a title"
print("Correct: 73 skin lines in 20 bins")

## 3. Two columns: a scatter plot

A scatter plot puts one column on the x axis and another on the y axis. Each dot is one row, here one cell line.

`s=` sets the size of the dots and `alpha=` makes them see-through (0 is invisible, 1 is solid), which helps when a thousand dots sit on top of each other.

In [ ]:
# DEMO - BRAF against SOX10, every cell line
fig, ax = plt.subplots()
ax.scatter(df["BRAF"], df["SOX10"], s=6, alpha=0.4)
ax.set_xlabel("BRAF gene effect")
ax.set_ylabel("SOX10 gene effect")
plt.show()

Most dots sit in a cloud near (0, 0). A tail runs down and to the left: cell lines that need **both** genes.

### Write it

**Your task:** draw the same scatter plot, BRAF on x and SOX10 on y, for the **skin** lines only. Label both axes.

In [ ]:
# TODO - BRAF against SOX10, skin only



# --- self-check (do not edit) ---
assert len(ax.collections) == 1, f"Your panel has {len(ax.collections)} scatter layers. Call ax.scatter once"
points = ax.collections[0].get_offsets()
assert len(points) == 73, f"Your scatter has {len(points)} dots. Plot the skin lines only"
assert np.allclose(points[:, 0], skin["BRAF"]), "The x values are not BRAF. The first argument of ax.scatter is x"
assert np.allclose(points[:, 1], skin["SOX10"]), "The y values are not SOX10"
assert "BRAF" in ax.get_xlabel() and "SOX10" in ax.get_ylabel(), "Label the x axis with BRAF and the y axis with SOX10"
print("Correct: 73 skin lines, BRAF on x, SOX10 on y")

## 4. Two groups on one panel

The skin-only plot has lost the comparison: you cannot see where skin sits **compared with** everything else. The fix from the lecture is to call `ax.scatter` twice on the same `ax`:

1. first the rest, in grey, as the background
2. then skin, in colour, on top

Give each call a `label=`. Then `ax.legend()` collects the labels into a key.

```python
ax.scatter(rest["BRAF"], rest["SOX10"], s=6, alpha=0.3, color="grey", label="other lineages")
ax.scatter(skin["BRAF"], skin["SOX10"], s=12, color="tab:blue", label="skin")
ax.legend()
```

### Write it

**Your task:** make that figure: the rest in grey, then the skin lines in colour, with labels `other lineages` and `skin`, a legend and both axis labels.

In [ ]:
# TODO - grey for the rest, colour for skin



# --- self-check (do not edit) ---
assert len(ax.collections) == 2, f"Your panel has {len(ax.collections)} scatter layers. Call ax.scatter twice"
first, second = [len(c.get_offsets()) for c in ax.collections]
assert (first, second) == (1092, 73), f"The layers have {first} and {second} dots. Draw rest (1,092) first, then skin (73)"
r, g, b, _ = ax.collections[0].get_facecolor()[0]
assert abs(r - g) < 0.05 and abs(g - b) < 0.05, "Make the first layer (the rest) grey: color='grey'"
legend = ax.get_legend()
assert legend is not None, "Add a legend with ax.legend()"
labels = [t.get_text() for t in legend.get_texts()]
assert labels == ["other lineages", "skin"], f"The legend says {labels}. Put label='...' in each ax.scatter call"
print("Correct: the skin lines stand out in the bottom left")

### Broken

This figure should show the skin lines in blue, but there is no blue to be seen. Nothing is wrong with the data. Look at the order of the two `ax.scatter` calls.

In [ ]:
# BROKEN - where are the skin lines?
fig, ax = plt.subplots()
ax.scatter(skin["BRAF"], skin["SOX10"], s=12, color="tab:blue", label="skin")
ax.scatter(df["BRAF"], df["SOX10"], s=20, color="grey", label="all lines")
ax.legend()
plt.show()


# --- self-check (do not edit) ---
assert len(ax.collections[-1].get_offsets()) == 73, "The last layer drawn should be the skin lines"
print("Fixed. Each call draws on top of the one before. df contains the skin lines too, so the grey dots covered every blue one.")

## 5. A box plot

A box plot summarises one column for several groups side by side. The line in the box is the **median**, the box holds the middle half of the values, and the whiskers reach out to the rest.

`ax.boxplot` takes a **list** of columns, one per box, and `tick_labels=` names the boxes. The lecture built the list with a list comprehension:

In [ ]:
# DEMO - SOX10 in four lineages
names = ["Skin", "Lung", "Bowel", "Breast"]
groups = [df[df["oncotree_lineage"] == n]["SOX10"] for n in names]

fig, ax = plt.subplots()
ax.boxplot(groups, tick_labels=names)
ax.set_ylabel("SOX10 gene effect")
plt.show()

### Write it

**Your task:** draw a box plot with **two** boxes: SOX10 in `skin`, and SOX10 in `rest`. Name them `skin` and `all others`, and label the y axis.

In [ ]:
# TODO - skin against all others



# --- self-check (do not edit) ---
names = [t.get_text() for t in ax.get_xticklabels()]
assert names == ["skin", "all others"], f"The boxes are named {names}. Use tick_labels=['skin', 'all others']"
medians = [line.get_ydata()[0] for line in ax.lines if len(line.get_ydata()) == 2 and line.get_ydata()[0] == line.get_ydata()[1]]
assert any(abs(m - skin["SOX10"].median()) < 1e-9 for m in medians), "There is no box for skin['SOX10']. Pass a list: [skin['SOX10'], rest['SOX10']]"
assert ax.get_ylabel() != "", "Label the y axis"
print("Correct: the skin box sits far below the rest, and it is tall")

## 6. Every cell line as a dot

The skin box is tall: the values are spread out. The box cannot tell you whether that is one wide group or two separate ones. Drawing every cell line as a dot on top of the box shows it.

Box number 1 sits at x = 1, box number 2 at x = 2, and so on. To put dots on box 1, give every dot an x value close to 1. If they were all exactly 1 they would sit in one line on top of each other, so add a little random sideways spread with `np.random.normal(1, 0.06, n)`: n numbers centred on 1.

In [ ]:
# DEMO - dots on one box
values = skin["SOX10"]
x = np.random.normal(1, 0.06, len(values))   # 73 numbers close to 1

fig, ax = plt.subplots()
ax.boxplot([values], tick_labels=["skin"], showfliers=False)
ax.scatter(x, values, s=10, alpha=0.6)
ax.set_ylabel("SOX10 gene effect")
plt.show()

`showfliers=False` turns off the box plot's own dots for unusual values, because you are now drawing every value yourself.

### Write it

**Your task:** draw the two-box plot from section 5 again (skin and all others, `showfliers=False`), and put dots on **both** boxes: the skin values at x close to 1, the rest at x close to 2.

In [ ]:
# TODO - two boxes, every cell line as a dot



# --- self-check (do not edit) ---
assert [t.get_text() for t in ax.get_xticklabels()] == ["skin", "all others"], "Name the boxes skin and all others"
sizes = [len(c.get_offsets()) for c in ax.collections]
assert sorted(sizes) == [73, 1092], f"Your dot layers have {sizes} dots. Draw skin (73) and rest (1,092)"
for layer in ax.collections:
    xs = layer.get_offsets()[:, 0]
    centre = 1 if len(xs) == 73 else 2
    assert abs(xs.mean() - centre) < 0.05, f"The {len(xs)} dots should sit around x = {centre}"
print("Correct: now you can see the skin lines that do not need SOX10")

Look at the skin dots. Most sit low, below -1, but a group sits up near 0, level with the other cancers. Those are the 18 lines from notebook 1. The mean, and even the box, made them easy to miss.

## 7. Many boxes with a loop

For two boxes you can write two `ax.scatter` lines. For six you want a loop. `enumerate(groups, start=1)` gives you each group **and** its position, counting from 1, which is the same as the x position of its box.

In [ ]:
# DEMO - what enumerate gives you
for i, name in enumerate(["Skin", "Lung", "Bowel"], start=1):
    print(i, name)

### Write it

**Your task:** draw a box plot of SOX10 for the six lineages in `names`, with `showfliers=False`, and use a `for` loop with `enumerate` to put every cell line on its box as a dot.

In [ ]:
# TODO - six boxes, dots from a loop
names = ["Skin", "Lung", "Lymphoid", "CNS/Brain", "Bowel", "Breast"]
groups = [df[df["oncotree_lineage"] == n]["SOX10"] for n in names]



# --- self-check (do not edit) ---
assert [t.get_text() for t in ax.get_xticklabels()] == names, "Use tick_labels=names"
sizes = [len(c.get_offsets()) for c in ax.collections]
assert sizes == [73, 124, 92, 89, 62, 53], f"Your dot layers have {sizes} dots. Draw one layer per group, in the same order as names"
centres = [round(c.get_offsets()[:, 0].mean()) for c in ax.collections]
assert centres == [1, 2, 3, 4, 5, 6], f"The dot layers sit around x = {centres}. Use enumerate(groups, start=1)"
print("Correct: six lineages, every cell line visible")

## 8. Saving the figure

`fig.savefig` writes the figure to a file you can put in a report. `dpi=300` makes it sharp enough to print, and `bbox_inches="tight"` trims the empty margin. On Colab the file appears in the folder panel on the left.

### Write it

**Your task:** make the two-box figure from section 6 again, give it a title, and save it as `sox10_skin.png` at 300 dpi.

In [ ]:
# TODO - make the figure and save it



# --- self-check (do not edit) ---
assert Path("sox10_skin.png").exists(), "There is no file called sox10_skin.png. Use fig.savefig('sox10_skin.png', dpi=300)"
assert ax.get_title() != "", "Give the figure a title before you save it"
print("Correct: saved as sox10_skin.png")

## If you finish early

1. Redraw the six-lineage box plot with the lineages **sorted by their median** SOX10, lowest first. (Hint: make a list of `(median, name)` pairs and sort it, or use `groupby` and `sort_values`.)
2. Colour the skin dots in your loop differently from the others. Inside the loop, use an `if` to pick the colour.
3. In the grey-and-blue scatter, add a third layer for the **Eye** lineage in another colour. Uveal melanoma grows in the eye. Where do those lines sit?

In [ ]:
# Your code here

## Recap

You can now:

- start any figure with `fig, ax = plt.subplots()` and label it with `ax.set_xlabel`, `ax.set_ylabel` and `ax.set_title`
- show one column with `ax.hist`, and two columns with `ax.scatter`
- draw a background group in grey and a group of interest on top in colour, with `label=` and `ax.legend()`
- compare groups with `ax.boxplot`, and show every value as a dot with `np.random.normal` for the sideways spread
- draw one layer per group with `enumerate(..., start=1)`
- save a figure with `fig.savefig(..., dpi=300)`

**Next:** `03_a_grid_and_a_loose_end`: several panels in one figure, and who the 18 skin lines are.